# AURA — Intent Classifier v2 (robust to spelling + out-of-scope class)

**What changed from v1**
1. **Spelling-variant augmentation**: each training message also appears re-spelled the way people type on WhatsApp
   (`hai→hy`, `cancel→cancle/cncl`, `mujhe→mujhy`, `appointment→appt`…).
   v1 dropped from **94.1% → 77.4%** on re-spelled test messages and read "cancle" as *booking* at 100% confidence.
2. **New class `other`**: thanks, "ok", names, off-topic questions. v1 had to force these into one of 6 intents
   (it got **0/23** right, mostly "greeting" at 99% confidence).

**Test sets** (none of these messages are in training):
- `test` — the original 136 messages (directly comparable with v1)
- `test_noisy` — the same messages re-spelled (robustness)
- `test_other` — out-of-scope messages
- `test_short` — short requests like "Booking karni hy", "rates?", "AOA"

### How to run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When asked, upload the **6 files** from `AURA/training/data_v2/`: `train.json`, `val.json`, `test.json`, `test_noisy.json`, `test_other.json`, `test_short.json`
4. Allow Google Drive access — the model is saved to `My Drive/AURA/intent_model_v2`

About 15–25 minutes.

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate scikit-learn

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

## 2. Upload the v2 dataset

In [ ]:
import json, os
from google.colab import files

FILES = ["train.json", "val.json", "test.json", "test_noisy.json", "test_other.json", "test_short.json"]
missing = [f for f in FILES if not os.path.exists(f)]
if missing:
    print("Please upload (you can select all 6 at once):", missing)
    files.upload()

data = {}
for f in FILES:
    with open(f, encoding="utf-8") as fh:
        data[f[:-5]] = json.load(fh)
    print(f"{f:16} {len(data[f[:-5]])} messages")

## 3. Labels and tokenizer

In [ ]:
from collections import Counter
from transformers import AutoTokenizer, set_seed

SEED, MODEL_NAME, MAX_LEN = 42, "xlm-roberta-base", 96
set_seed(SEED)

LABELS = sorted({r["intent"] for r in data["train"]})
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print("Labels:", LABELS)
print("train:", dict(Counter(r["intent"] for r in data["train"])))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.enc = tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LEN)
        self.labels = [label2id[r["intent"]] for r in rows]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

ds = {name: IntentDataset(rows) for name, rows in data.items()}

## 4. Train

In [ ]:
import math
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, EarlyStoppingCallback)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(pred):
    y_pred = np.argmax(pred.predictions, axis=-1)
    return {"accuracy": accuracy_score(pred.label_ids, y_pred),
            "macro_f1": f1_score(pred.label_ids, y_pred, average="macro")}

EPOCHS, BATCH = 8, 16
warmup_steps = int(0.1 * math.ceil(len(ds["train"]) / BATCH) * EPOCHS)   # works on transformers v4 and v5

args = TrainingArguments(
    output_dir="checkpoints_v2",
    num_train_epochs=EPOCHS,
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"],
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer.train()
print("Best validation:", trainer.evaluate())

## 5. Evaluate on the three test sets

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

ALL_IDS = list(range(len(LABELS)))
results, preds = {}, {}
for name in ["test", "test_noisy", "test_other", "test_short"]:
    out = trainer.predict(ds[name])
    probs = torch.softmax(torch.tensor(out.predictions), dim=-1).numpy()
    preds[name] = (out.label_ids, probs.argmax(-1), probs.max(-1))
    y_true, y_pred, _ = preds[name]
    results[name] = {"n": len(y_true), "accuracy": accuracy_score(y_true, y_pred),
                     "macro_f1": f1_score(y_true, y_pred, labels=sorted(set(y_true)), average="macro") if name != "test_other" else float("nan")}
    print(f"{name:11} n={len(y_true):4}  accuracy={results[name]['accuracy']:.3f}  macro-F1={results[name]['macro_f1']:.3f}")

y_true, y_pred, _ = preds["test"]
present = sorted(set(y_true))   # macro-F1 only over intents that occur in this test set
report = classification_report(y_true, y_pred, labels=present, target_names=[LABELS[i] for i in present], digits=3, zero_division=0)
print("\nOriginal test set, per intent:\n", report)

# Confusion matrix over original test + out-of-scope test
yt = np.concatenate([preds["test"][0], preds["test_other"][0]])
yp = np.concatenate([preds["test"][1], preds["test_other"][1]])
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_matrix(yt, yp, labels=ALL_IDS), display_labels=LABELS).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title("XLM-RoBERTa v2 — test + out-of-scope")
plt.tight_layout(); plt.savefig("confusion_matrix_v2.png", dpi=150); plt.show()

errors = []
for name in ["test", "test_noisy", "test_other", "test_short"]:
    t, p, c = preds[name]
    for i in range(len(t)):
        if t[i] != p[i]:
            errors.append({"set": name, "text": data[name][i]["text"], "true": id2label[int(t[i])],
                           "predicted": id2label[int(p[i])], "confidence": round(float(c[i]), 3)})
print(f"{len(errors)} mistakes in total:")
for e in errors:
    print(f"  [{e['set']}] {e['true']} -> {e['predicted']} @ {e['confidence']}: {e['text']}")

### Confidence threshold
AURA asks Gemini when the model is less sure than the threshold (currently 0.70).

In [ ]:
t = np.concatenate([preds[n][0] for n in preds]); p = np.concatenate([preds[n][1] for n in preds])
c = np.concatenate([preds[n][2] for n in preds])
print(f"{'threshold':>9} | {'answered by model':>17} | {'accuracy on those':>17}")
for th in [0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = c >= th
    print(f"{th:>9} | {keep.mean():>16.1%} | {accuracy_score(t[keep], p[keep]):>17.3f}")

## 6. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
assert os.path.ismount("/content/drive"), "Google Drive is NOT connected - run this cell again and click Allow"

SAVE_DIR = "/content/drive/MyDrive/AURA/intent_model_v2"
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!cp confusion_matrix_v2.png "$SAVE_DIR/"
with open(f"{SAVE_DIR}/results_v2.json", "w") as f:
    json.dump(results, f, indent=2)
with open(f"{SAVE_DIR}/errors_v2.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)
with open(f"{SAVE_DIR}/classification_report_v2.txt", "w") as f:
    f.write(report)
# Check the 1.1 GB weights file really exists before going on
weights = os.path.join(SAVE_DIR, "model.safetensors")
size_gb = os.path.getsize(weights) / 1e9 if os.path.exists(weights) else 0
assert size_gb > 0.5, f"model.safetensors missing or too small ({size_gb:.2f} GB) - re-run this cell"
print(f"Saved to {SAVE_DIR}  (model.safetensors = {size_gb:.2f} GB)")

# Reload the SAVED copy and score it: catches saving an untrained model (e.g. after a Colab reset)
_m = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).eval().to(model.device)
_ok = 0
with torch.no_grad():
    for r in data["test"]:
        enc = tokenizer(r["text"], return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        _ok += id2label[int(_m(**enc).logits.argmax(-1))] == r["intent"]
saved_acc = _ok / len(data["test"])
print(f"Saved model accuracy on test: {saved_acc:.3f}")
assert saved_acc > 0.85, "The SAVED model is not the trained one - re-run Runtime > Run all"
del _m
!ls -lh "$SAVE_DIR"

## 7. v1 vs v2 (fill v1 numbers from `python training/evaluate_model.py models/intent_model`)

In [ ]:
import pandas as pd
V1 = {"test": 0.941, "test_noisy": 0.774, "test_other": 0.000, "test_short": float("nan")}     # measured locally on 12 Nov
table = pd.DataFrame([{"Test set": n, "v1 accuracy": V1[n], "v2 accuracy": round(results[n]["accuracy"], 3)}
                      for n in ["test", "test_noisy", "test_other", "test_short"]])
display(table)
table.to_csv(f"{SAVE_DIR}/v1_vs_v2.csv", index=False)

## 8. Try your own messages

In [ ]:
cpu_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).eval()
cpu_tok = AutoTokenizer.from_pretrained(SAVE_DIR)

def predict(text):
    with torch.no_grad():
        pr = torch.softmax(cpu_model(**cpu_tok(text, return_tensors="pt")).logits, -1)[0]
    i = int(pr.argmax())
    return id2label[i], round(float(pr[i]), 3)

for msg in ["Booking karni hy", "booking cancle krni he", "Shukriya", "Jazib umer", "Timings",
            "ok", "kal 3 bje haircut", "rates kya hain", "Assalam o alaikum", "appt shft kr dein friday ko"]:
    print(predict(msg), "<-", msg)

## 9. Finish uploading to Google Drive (important)
Colab uploads to Drive in the background. A 1.1 GB file takes a few minutes, so this cell waits
until everything is uploaded. **Only download the folder from Drive after this cell prints DONE.**

In [ ]:
from google.colab import drive
print('Uploading to Google Drive, please wait (can take 5-10 minutes)...')
import os
assert os.path.ismount('/content/drive'), 'Google Drive is not connected - re-run section 6 first'
drive.flush_and_unmount()
print('DONE - all files are in My Drive/AURA/intent_model_v2. You can download the folder now.')
